# **🇵🇰 Urdu Sentiment Analysis using mBERT**

##### Fine-tuning a Multilingual Transformer for Low-Resource Urdu NLP


###  1. **Problem Statement**

Urdu is a **low-resource language** in Natural Language Processing (NLP), with limited labeled datasets and pretrained models.

This makes sentiment analysis for Urdu challenging in real-world applications.

#### This project aims to:

* Develop a robust Urdu sentiment classification system
* Fine-tune a transformer-based model on Urdu dataset
* Build an end-to-end deployable AI system
* Enable real-time sentiment prediction through a web interface


### 2. **Project Overview**

This project focuses on building an **Urdu sentiment analysis system** using a fine-tuned Multilingual BERT model **mBERT**.

####  **Objective**:

The goal is to classify Urdu text into three sentiment categories:

*  Positive
*  Neutral
*  Negative


####  **Real-world Use-cases**:

* Social media sentiment analysis
* Customer feedback systems
* Product review classification
* Urdu NLP research applications



###  3. **Dataset**

* Source: HuggingFace Dataset Hub
* Dataset: Urdu Multi-Domain Classification

🔗 [https://huggingface.co/datasets/umar178/UrduMultiDomainClassification](https://huggingface.co/datasets/umar178/UrduMultiDomainClassification)


###  4. **Data Preprocessing**

To prepare the dataset for training, the following steps were performed:

### ✔ Cleaning steps:

* Removed irrelevant columns `topic`, `intent`, `binary`
* Filtered dataset to focus on 3 main sentiment classes
* Removed low-frequency class `request`
* Ensured balanced class distribution


###  5. **Text Normalization**

Urdu text is normalized using **LughaatNLP** to improve model performance.

### ✔ Normalization helps in:

* Removing noise and inconsistencies
* Standardizing Urdu spelling variations
* Improving tokenization quality
* Enhancing model generalization


### 6. **Project Pipeline Overview**


```text
Raw Urdu Text
      ↓
Normalization (LughaatNLP)
      ↓
Tokenization (mBERT)
      ↓
Fine-tuned Transformer Model
      ↓
Sentiment Prediction (Positive / Negative / Neutral)
      ↓
Gradio Web Interface
```


### **Importing Libraries**

In [ ]:
# !pip install datasets
# !pip install lughaatNLP
# !pip install --upgrade torch
# # !pip install transformers
# !pip uninstall -y torch torchvision torchaudio
# !pip install torch torchvision torchaudio



In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch, os
from transformers import BertTokenizerFast, BertForSequenceClassification
from transformers import pipeline
from torch.utils.data import Dataset
from torch import cuda
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from datasets import Dataset
from transformers import DataCollatorWithPadding
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import TrainingArguments, Trainer
from google.colab import files
import shutil

### **Data Loading**

In [ ]:
dataset = load_dataset("umar178/UrduMultiDomainClassification")

In [ ]:
print(dataset)
print(dataset["train"][0])

convert to Pandas DataFrame

In [ ]:

df = dataset["train"].to_pandas()
df.head()

### **Data Cleaning & Preprocessing**

In [ ]:
df = df.drop(columns=["topic", "intent", 'binary'])
df.head()

In [ ]:
df.shape

In [ ]:
df['sentiment'].value_counts()

In [ ]:
df.shape

In [ ]:
34819 - 188

In [ ]:
df = df[df["sentiment"] != "request"]

In [ ]:
df.shape

In [ ]:
df['sentiment'].value_counts()

In [ ]:
df

In [ ]:
# save the data

df.to_csv("urdu_text_classifier_data.csv", index=False)

In [ ]:

device = 'cuda' if cuda.is_available() else 'cpu'
device

In [ ]:
df = pd.read_csv('/content/urdu_text_classifier_data.csv')
df.head()

### **Urdu Text Normalization**

In [ ]:
from LughaatNLP import LughaatNLP


urdu_text_processing = LughaatNLP()

df['normalized_text'] = df['text'].apply(
    lambda x: urdu_text_processing.normalize(x)
)


df.head()

### **Data Encoding**

In [ ]:
labels = df['sentiment'].unique().tolist()
labels

In [ ]:
for key, value in enumerate(labels):
  print(value)

In [ ]:
NUM_LABELS = len(labels)
ID_TO_LABEL = { id: label for id, label in enumerate(labels)}
LABEL_TO_ID = { label: id for id, label in enumerate(labels)}

In [ ]:
ID_TO_LABEL

In [ ]:
LABEL_TO_ID

In [ ]:
df['labels'] = df['sentiment'].map(LABEL_TO_ID)
df.head()

### **Exploratory Data Analysis**

In [ ]:
df['sentiment'].value_counts().plot(kind='pie', figsize=(8,8))

In [ ]:
df['sentiment'].value_counts()

### **Model Initialization & Tokenization**

> Use the Multilingual BERT:

Why mBERT?
- Supports Urdu language
- Pretrained on multilingual corpus
- Strong performance on low-resource languages

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')

In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=NUM_LABELS, id2label=ID_TO_LABEL, label2id=LABEL_TO_ID)
model.to(device)

### **Data Preparation for Training**

In [ ]:
filtered_df = df[['normalized_text', 'labels']]

filtered_df.head()

In [ ]:

dataset = Dataset.from_pandas(filtered_df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)


In [ ]:
dataset

In [ ]:
temp = dataset["test"].train_test_split(test_size=0.5)


In [ ]:
temp

In [ ]:
train_dataset = dataset["train"]
val_dataset = temp["train"]
test_dataset = temp["test"]

In [ ]:
len(train_dataset), len(val_dataset), len(test_dataset)

In [ ]:
train_dataset

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["normalized_text"],
        truncation=True,
        padding=True
    )

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer)

### **Evaluation Metrics**

In [ ]:

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


### **Training Configuration**

In [ ]:
train_arguments = TrainingArguments(
    output_dir='./results',
    do_train=True,
    do_eval=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    eval_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=7,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
)

### **Model Training**

In [ ]:
trainer = Trainer(
    model = model,

    args = train_arguments,

    train_dataset = train_dataset,

    eval_dataset = val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

### **Model Saving**

In [ ]:
trainer.save_model("./best_model")
tokenizer.save_pretrained("./best_model")

In [ ]:

shutil.make_archive("best_model", "zip", "./best_model")
files.download("best_model.zip")

### **Model Evaluation**

In [ ]:
trainer.evaluate()

In [ ]:
pred_output = trainer.predict(test_dataset)

In [ ]:
pred_output

In [ ]:

y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

#### **Confusion Matrix Analysis**

In [ ]:


cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:


sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:


print(classification_report(y_true, y_pred))

### **Inference Pipeline**

In [ ]:


model_path = "./best_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
model.config.id2label = ID_TO_LABEL
model.config.label2id = LABEL_TO_ID

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    pred_id = torch.argmax(probs, dim=1).item()
    confidence = torch.max(probs).item()

    label = model.config.id2label[pred_id]

    return label, confidence

In [ ]:
print(predict_sentiment("استاد کا پڑھانے کا انداز بہت اچھا ہے"))

In [ ]:

# print(predict_sentiment("یہ بہت برا تجربہ تھا"))


print(predict_sentiment("ایپ بار بار کریش ہو رہی ہے، بہت مسئلہ ہے"))

In [ ]:

print(predict_sentiment("وہ بازار گیا اور سامان خریدا۔"))

###  **Deployment**

The model is deployed using:

- Gradio UI
- HuggingFace Spaces

This allows real-time Urdu sentiment prediction.

<br>

## **Conclusion**

This project successfully demonstrates:

- Fine-tuning of a transformer model for Urdu NLP
- Effective preprocessing for low-resource language
- Deployment of AI system using Gradio + HuggingFace
-  End-to-End NLP System from Training to Deployment



> This system can be used for:
- Social media analysis
- Customer feedback systems
- Urdu review classificatio